# Part 5: Fine-tuning with a loop you can read

**Fine-tuning** continues training a pretrained model on your own labelled data. You already know every line of the loop: this is the Module 07 loop with a pretrained body and a new head. Each cell below that trains takes about two minutes on a CPU.

In [ ]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 200, seed=42)

## What changes in the model

The body is pretrained. The two-layer head is new and starts with random weights, which is why Hugging Face prints a warning about newly initialized weights. That is expected.

In [ ]:
from transformers import AutoModelForSequenceClassification

from finetune import MODEL, freeze_body

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
total = sum(p.numel() for p in model.parameters())
print("all parameters", total)
print("head only", freeze_body(model))

## Train only the head

The body is frozen, so a larger learning rate is safe.

In [ ]:
from finetune import evaluate, fine_tune

model, tokenizer, history = fine_tune(
    texts, labels, val_texts, val_labels, head_only=True, lr=1e-3
)
for row in history:
    print(row)

## Train everything

Now the pretrained layers move too, so the learning rate must be small. A large one would wreck what the model already knows.

In [ ]:
model, tokenizer, history = fine_tune(texts, labels, val_texts, val_labels, lr=5e-5)
for row in history:
    print(row)

In [ ]:
print(evaluate(model, tokenizer, val_texts, val_labels))

## Try it

1. Train the whole model with a learning rate of `5e-3`. What happens to the validation accuracy?
2. Train for 6 epochs instead of 3. Does the training loss keep falling? Does the validation accuracy?
3. Train the head only for 3 epochs at `5e-5`. Why is it so much worse than at `1e-3`?